# Batch Dynamics Check — Single City

This notebook loads a single city's delivery CSV, sorts by `delivery_user_id` and `receipt_time`, computes batch features (batch size, dispatch rank, batch id) and checks for `dynamic pickup` events: where a courier receives a new batch while some orders from the previous batch remain undelivered (sign_time > next_batch_receipt_time).

Edit `CITY` and `CITY_CSV_PATHS` below to point to your local files if needed.

In [ ]:
# Imports and config
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd

# Select city to analyse (case-sensitive, e.g. 'Shanghai')
CITY = 'Shanghai'

# Candidate paths (edit if your files live elsewhere)
CITY_CSV_PATHS = [
    './city_divided/shanghai_data.csv',
    './city_divided/shanghai_data.csv',
    '/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv',
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

CITY_CSV = find_existing_path(CITY_CSV_PATHS)
if CITY_CSV is None:
    print('No candidate CSV found. Please update CITY_CSV_PATHS to point to your file.')
    sys.exit(1)

print('Using CSV:', CITY_CSV)


In [ ]:
# Load the CSV with pandas (robust for various environments)
df = pd.read_csv(CITY_CSV)

# Ensure datetime columns parsed
for col in ['receipt_time', 'sign_time']:
    if col in df.columns and not np.issubdtype(df[col].dtype, np.datetime64):
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Basic cleaning: drop rows missing key timestamps or courier id
df = df.dropna(subset=['receipt_time', 'delivery_user_id'])
df['delivery_user_id'] = df['delivery_user_id'].astype(str)

# Compute eta_mins if present/needed
if 'sign_time' in df.columns:
    df['eta_mins'] = (df['sign_time'] - df['receipt_time']).dt.total_seconds() / 60

# Sort as requested: by delivery_user_id then receipt_time then order_id (if exists)
sort_cols = ['delivery_user_id', 'receipt_time']
if 'order_id' in df.columns:
    sort_cols.append('order_id')
df = df.sort_values(sort_cols).reset_index(drop=True)

print('Loaded rows:', len(df))
df.head(3)


In [ ]:
# Compute batch features: batch_size, batch_id, batch_rank_dispatch
# Definition: orders sharing (delivery_user_id, receipt_time) are a batch
group_cols = ['delivery_user_id', 'receipt_time']
df['batch_size'] = df.groupby(group_cols)['order_id'].transform('count') if 'order_id' in df.columns else df.groupby(group_cols).transform('size')
# batch_id: courier__epoch(receipt_time)
df['batch_id'] = df['delivery_user_id'] + '__' + df['receipt_time'].astype('int64').floordiv(10**9).astype(str)
# dispatch rank: deterministic order within the simultaneous push — use order_id if available else row order
if 'order_id' in df.columns:
    df['batch_rank_dispatch'] = df.groupby(group_cols)['order_id'].cumcount()
else:
    df['batch_rank_dispatch'] = df.groupby(group_cols).cumcount()

# actual rank by sign_time (post-hoc), stored only for diagnostics if sign_time exists
if 'sign_time' in df.columns:
    df['batch_rank_actual'] = df.groupby(group_cols)['sign_time'].rank(method='first').astype('Int64') - 1
else:
    df['batch_rank_actual'] = pd.NA

print('Batch features computed. Example:')
display(df[['delivery_user_id','receipt_time','batch_id','batch_size','batch_rank_dispatch','batch_rank_actual']].head(8))


In [ ]:
# Check for dynamic pickup: for each courier, for each batch, did a next batch arrive while some
# orders in the current batch were still undelivered? (i.e., sign_time > next_batch_receipt_time)

# We need sign_time for this diagnostic; if absent, report and stop
if 'sign_time' not in df.columns:
    print('sign_time column missing — cannot compute dynamic pickup. Add sign_time to CSV and retry.')
else:
    # Build per-courier batch summary sorted by receipt_time
    batches = df.groupby(['delivery_user_id','batch_id'], as_index=False).agg({
        'receipt_time': 'first',
        'batch_size': 'first',
        'order_id': lambda s: list(s) if 'order_id' in df.columns else None,
    })
    batches = batches.sort_values(['delivery_user_id','receipt_time']).reset_index(drop=True)

    # Map next receipt_time per courier (shift)
    batches['next_batch_receipt'] = batches.groupby('delivery_user_id')['receipt_time'].shift(-1)

    # For each batch, count unfinished orders at next_batch_receipt
    def count_unfinished(row):
        if pd.isna(row['next_batch_receipt']):
            return 0
        mask = (df['batch_id'] == row['batch_id']) & (df['sign_time'] > row['next_batch_receipt'])
        return int(mask.sum())

    batches['unfinished_at_next'] = batches.apply(count_unfinished, axis=1)
    batches['dynamic_pickup_flag'] = (batches['unfinished_at_next'] > 0).astype(int)

    # Summary statistics
    total_batches = len(batches)
    dyn_count = int(batches['dynamic_pickup_flag'].sum())
    print(f'Total batches (detected): {total_batches:,}')
    print(f'Batches with dynamic pickup (new batch arrived while previous had unfinished deliveries): {dyn_count:,} ({dyn_count/total_batches*100:.2f} %)')

    # Distribution by batch size
    size_tab = batches.groupby('batch_size')['dynamic_pickup_flag'].agg(['count','sum']).reset_index()
    size_tab['pct_dynamic'] = size_tab['sum'] / size_tab['count'] * 100
    display(size_tab.sort_values('batch_size').head(20))

    # Show example cases where dynamic pickup happened
    examples = batches[batches['dynamic_pickup_flag'] == 1].head(10)
    if not examples.empty:
        for _, r in examples.iterrows():
            print('
Example courier:', r['delivery_user_id'], 'receipt_time:', r['receipt_time'])
            display(df[df['batch_id'] == r['batch_id']][['order_id','receipt_time','sign_time','batch_rank_dispatch','batch_rank_actual']].sort_values('batch_rank_dispatch'))
    else:
        print('No dynamic pickup examples found in the sampled data.')


In [ ]:
# Optional: save batch-level summary for further analysis
OUT = './batch_dynamics_summary_' + CITY.lower() + '.csv'
batches.to_csv(OUT, index=False)
print('Saved batch summary to', OUT)


## Notes
- This notebook treats orders with identical `receipt_time` (same courier) as part of the same batch, matching the LaDe pipeline's definition.
- Dynamic pickup is detected when a later batch's `receipt_time` occurs while some orders in the earlier batch remain unsigned (`sign_time > next_batch_receipt_time`).
- If `sign_time` is missing or inaccurate, consider using courier GPS traces or delivery status logs to refine the overlap definition.

If you want, I can run this notebook for a specific city file in your environment, or extend it to visualise temporal overlap per courier.